In [1]:
import torch
from torch import nn
import plotly.graph_objects as go
from edl_pytorch import NormalInvGamma
from edl_pytorch.loss import nig_nll, nig_reg

torch.manual_seed(0)


def generate_data(func, x_min_train, x_max_train, num_train=1000, num_test=1000, sigma_scale=3, test_additional=0.4):
    # Training data
    x_train = torch.linspace(x_min_train, x_max_train, num_train).unsqueeze(-1)
    sigma = torch.normal(torch.zeros_like(x_train), sigma_scale * torch.ones_like(x_train))
    y_train = func(x_train) + sigma

    # Automatically set the test range based on the ratio (test_add)
    test_range = abs(x_max_train - x_min_train) * test_additional
    x_min_test = x_min_train - test_range
    x_max_test = x_max_train + test_range

    # Test data
    x_test = torch.linspace(x_min_test, x_max_test, num_test).unsqueeze(-1)
    y_test = func(x_test)
    
    return x_train, y_train, x_test, y_test


# Uncertainty Regularization to Bypass HUA
# From https://arxiv.org/abs/2401.01484
# Based on the equation from the paper (equation 17)
def u_reg(gamma, _v, alpha, _beta, y):
    error = (y - gamma).abs()  # |y - gamma|
    uncertainty = torch.log(torch.exp(alpha - 1) - 1)  # log(exp(alpha - 1) - 1)
    reg = -error * uncertainty  # Combine the terms as shown in the formula
    return reg.mean()  # Return the mean value of the regularization term


# Non-saturating Uncertainty Regularizer
# From https://ojs.aaai.org/index.php/AAAI/article/view/30172:
# Based on the equation from the paper (equation 20)
def nsu_reg(gamma, v, alpha, beta, y):
    # (y - gamma)^2 * (v * (alpha - 1)) / (beta * (v + 1))
    error = (y - gamma).pow(2)
    uncertainty = v * (alpha - 1) / (beta * (v + 1))
    reg = error * uncertainty
    return reg.mean()


# Lipschitz Modified MSE Regularization
# Implemented based on the equation in the paper (equation 6)
def lipschitz_mse_reg(gamma, v, alpha, beta, y):
    # Calculate the squared error (y - gamma)^2
    error = (y - gamma).pow(2)
    
    # Calculate U_alpha and U_nu based on the given formula
    U_nu = beta * (1 + v) / (alpha * v)
    U_alpha = 2 * beta * (1 + v) / v * (torch.exp(torch.digamma(alpha + 0.5) - torch.digamma(alpha)) - 1)

    # Minimum U_nu and U_alpha value to control the Lipschitz constant
    U_nu_alpha = torch.min(U_nu, U_alpha)

    # Lipschitz Modified MSE: switch between normal MSE and Lipschitz adjusted term
    modified_mse = torch.where(
        error < U_nu_alpha, 
        error, 
        2 * torch.sqrt(U_nu_alpha) * torch.abs(y - gamma) - U_nu_alpha
    )

    return modified_mse.mean()


# HOVR regularization adapted for the EvidentialMLP model
def hovr_reg(model, x, k=3, q=2, num_points=10):
    # Generate random points within the input range
    x_min, x_max = x.min(0)[0], x.max(0)[0]
    random_points = torch.tensor(
        np.random.uniform(x_min.numpy(), x_max.numpy(), (num_points, x.shape[1])),
        dtype=torch.float32,
        requires_grad=True,
    )

    # Compute the model's output for the random points (mu only)
    mu, _, _, _ = model(random_points)  # Model returns a tuple (mu, v, alpha, beta)

    # Compute gradients of the output with respect to the random points
    grads = torch.autograd.grad(
        mu, random_points, torch.ones_like(mu), create_graph=True
    )[0]

    # Calculate HOVR term based on the gradients
    hovr_term = 0.0
    for i in range(x.shape[1]):  # Calculate k-th order derivative for each dimension
        grad_i = grads[:, i]
        temp_grad = grad_i
        for _ in range(k - 1):
            temp_grad = torch.autograd.grad(
                temp_grad, random_points, torch.ones_like(temp_grad), create_graph=True
            )[0][:, i]
        hovr_term += torch.sum(torch.abs(temp_grad) ** q)

    return hovr_term / x.shape[1]  # Normalize by the input dimension


class EvidentialMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_units=[64, 64, 64],
        activations=["tanh", "tanh", "tanh"],
        l2_coeff=0.0,
        nig_coeff=1e-2,
        lipschitz_coeff=0.0,
        nsu_coeff=0.0,
        hovr_coeff=0.0,
        epoch_num=500,
        lr=5e-4
    ):
        super(EvidentialMLP, self).__init__()
        layers = []
        in_dim = input_dim

        # Build layers based on the given hidden_units and activations
        for hidden_dim, activation in zip(hidden_units, activations):
            layers.append(nn.Linear(in_dim, hidden_dim))
            if activation == "relu":
                layers.append(nn.ReLU())
            elif activation == "tanh":
                layers.append(nn.Tanh())
            elif activation == "sigmoid":
                layers.append(nn.Sigmoid())
            in_dim = hidden_dim

        layers.append(NormalInvGamma(hidden_units[-1], output_dim))
        self.model = nn.Sequential(*layers)

        # Set coefficients for loss components
        self.l2_coeff = l2_coeff
        self.nig_coeff = nig_coeff
        self.lipschitz_coeff = lipschitz_coeff
        self.nsu_coeff = nsu_coeff
        self.hovr_coeff = hovr_coeff

        # Optimizer parameters
        self.epoch_num = epoch_num
        self.lr = lr

    def forward_dist_params(self, x):
        return self.model(x)

    def forward(self, x):
        with torch.no_grad():
            dist_params = self.forward_dist_params(x)
            mu, v, alpha, beta = (d.squeeze() for d in dist_params)
            std = torch.sqrt(beta / (v * (alpha - 1)))
        return mu, std
    
    def compute_loss(self, x, y):
        dist_params = self.forward_dist_params(x)

        loss = nig_nll(*dist_params, y)

        if self.nig_coeff > 0:
            loss += self.nig_coeff * nig_reg(*dist_params, y)
        
        if self.lipschitz_coeff > 0:
            loss += self.lipschitz_coeff * lipschitz_mse_reg(*dist_params, y)

        if self.nsu_coeff > 0:
            loss += self.nsu_coeff * nsu_reg(*dist_params, y)

        return loss
    
    def fit(self, x_train, y_train):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
    
        for epoch in range(self.epoch_num):
            # Full batch: x_train and y_train used as a whole
            loss = self.compute_loss(x_train, y_train)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0 or epoch == self.epoch_num - 1:
                print(f'Epoch: {epoch}, Loss: {loss.item()}')

In [3]:
import torch
from torch import nn
import plotly.graph_objects as go
from edl_pytorch import NormalInvGamma
from edl_pytorch.loss import nig_nll, nig_reg


torch.manual_seed(0)

# Define the true function
true_func = lambda x: torch.sin(2 * x) * x 

# Generate train and test data
x_train, y_train, x_test, y_test = generate_data(
    true_func, 
    x_min_train=4, 
    x_max_train=-4,
    num_train=100,
    num_test=1000,
    sigma_scale=1,
    test_additional=0,
)

# Create Evidential MLP model
model = EvidentialMLP(
    input_dim=1, 
    output_dim=1,
    hidden_units=[64, 64, 64],
    activations=["tanh", "tanh", "tanh"],
    nig_coeff=1e-2,
    lipschitz_coeff=0,
    nsu_coeff=0,
    epoch_num=500,
    lr=5e-4
)

# Train the model
model.fit(x_train, y_train)

# Make predictions on test data
mean, std = model(x_test)

# Prepare data for visualization
mean, std = mean.squeeze().numpy(), std.squeeze().numpy()
x_test, y_test = x_test.squeeze().numpy(), y_test.squeeze().numpy()

# Plot using Plotly
fig = go.Figure()

# Add train data points
fig.add_trace(go.Scatter(x=x_train.squeeze().numpy(), y=y_train.squeeze().numpy(), mode='markers', 
                         marker=dict(size=3, color='blue'), name='Train'))

# Add true function line
fig.add_trace(go.Scatter(x=x_test, y=y_test, mode='lines', 
                         line=dict(color='black', width=2), name='True'))

# Add predicted mean line
fig.add_trace(go.Scatter(x=x_test, y=mean, mode='lines', 
                         line=dict(color='blue', dash='dash'), name='Pred'))

# Add uncertainty (2 std devs only)
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean - 2 * std), 
    fill=None, mode='lines', line_color='rgba(0,0,255,0.1)',  # Lighter color
    showlegend=True, name='Unc. (2σ)'
))
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean + 2 * std), 
    fill='tonexty', mode='lines', line_color='rgba(0,0,255,0.1)',
    showlegend=False
))

# Update layout with white background
fig.update_layout(
    title="Evidential Regression",
    xaxis_title="x",
    yaxis_title="y",
    legend=dict(x=0, y=1),
    plot_bgcolor='white',  # White background
    paper_bgcolor='white'  # White background
)

fig.show()

Epoch: 0, Loss: 2.3764941692352295
Epoch: 10, Loss: 2.278574228286743
Epoch: 20, Loss: 2.229660987854004
Epoch: 30, Loss: 2.2019295692443848
Epoch: 40, Loss: 2.180161952972412
Epoch: 50, Loss: 2.1526834964752197
Epoch: 60, Loss: 2.1252076625823975
Epoch: 70, Loss: 2.103121519088745
Epoch: 80, Loss: 2.0915262699127197
Epoch: 90, Loss: 2.083468198776245
Epoch: 100, Loss: 2.0756373405456543
Epoch: 110, Loss: 2.0683083534240723
Epoch: 120, Loss: 2.0607805252075195
Epoch: 130, Loss: 2.0527029037475586
Epoch: 140, Loss: 2.0436949729919434
Epoch: 150, Loss: 2.03328013420105
Epoch: 160, Loss: 2.020752429962158
Epoch: 170, Loss: 2.0054023265838623
Epoch: 180, Loss: 1.9862496852874756
Epoch: 190, Loss: 1.9621583223342896
Epoch: 200, Loss: 1.9316169023513794
Epoch: 210, Loss: 1.8924168348312378
Epoch: 220, Loss: 1.8417526483535767
Epoch: 230, Loss: 1.7806490659713745
Epoch: 240, Loss: 1.705407738685608
Epoch: 250, Loss: 1.6246013641357422
Epoch: 260, Loss: 1.5461130142211914
Epoch: 270, Loss: 1.4

In [4]:
import torch
import numpy as np

def generate_data(
    func,
    x_min_train,
    x_max_train,
    num_train=100,
    num_test=1000,
    sigma_scale=0.2,
    test_additional=0.4,
    outlier_ratio=0.01,
    outlier_value_range=(5.0, 5.1),
):
    # Training data
    x_train = torch.linspace(x_min_train, x_max_train, num_train).unsqueeze(-1)
    sigma = torch.normal(
        torch.zeros_like(x_train), sigma_scale * torch.ones_like(x_train)
    )
    y_train = func(x_train) + sigma

    # Adding outliers to the training data
    n_outliers = int(num_train * outlier_ratio)
    outlier_indices = np.random.choice(num_train, n_outliers, replace=False)

    # Make sure the assigned outlier values match the shape of y_train
    y_train[outlier_indices] = outlier_value_range[0] + torch.rand(
        n_outliers
    ).unsqueeze(-1) * (outlier_value_range[1] - outlier_value_range[0])

    # Automatically set the test range based on the ratio (test_additional)
    test_range = abs(x_max_train - x_min_train) * test_additional
    x_min_test = x_min_train - test_range
    x_max_test = x_max_train + test_range

    # Test data
    x_test = torch.linspace(x_min_test, x_max_test, num_test).unsqueeze(-1)
    y_test = func(x_test)

    return x_train, y_train, x_test, y_test, outlier_indices

In [54]:
# Example usage
x_train, y_train, x_test, y_test, outlier_indices = generate_data(
    lambda x: torch.sin(2 * x) * x ,
    0,
    2 * torch.pi,
    num_train=30,
    test_additional=0,
    num_test=1000,
    outlier_ratio=0.01,
    outlier_value_range=(5.0, 5.1),
)


# Create Evidential MLP model
model = EvidentialMLP(
    input_dim=1, 
    output_dim=1,
    hidden_units=[64, 64, 64],
    activations=["tanh", "tanh", "tanh"],
    nig_coeff=1e-2,
    lipschitz_coeff=0,
    nsu_coeff=1e-1,
    # epoch_num=500,
    epoch_num=50,
    lr=5e-4
)

# Train the model
model.fit(x_train, y_train)

# Make predictions on test data
mean, std = model(x_test)

# Prepare data for visualization
mean, std = mean.squeeze().numpy(), std.squeeze().numpy()
x_test, y_test = x_test.squeeze().numpy(), y_test.squeeze().numpy()

# Plot using Plotly
fig = go.Figure()

# Add train data points
fig.add_trace(go.Scatter(x=x_train.squeeze().numpy(), y=y_train.squeeze().numpy(), mode='markers', 
                         marker=dict(size=3, color='blue'), name='Train'))

# Add true function line
fig.add_trace(go.Scatter(x=x_test, y=y_test, mode='lines', 
                         line=dict(color='black', width=2), name='True'))

# Add predicted mean line
fig.add_trace(go.Scatter(x=x_test, y=mean, mode='lines', 
                         line=dict(color='blue', dash='dash'), name='Pred'))

# Add uncertainty (2 std devs only)
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean - 2 * std), 
    fill=None, mode='lines', line_color='rgba(0,0,255,0.1)',  # Lighter color
    showlegend=True, name='Unc. (2σ)'
))
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean + 2 * std), 
    fill='tonexty', mode='lines', line_color='rgba(0,0,255,0.1)',
    showlegend=False
))

# Update layout with white background
fig.update_layout(
    title="Evidential Regression",
    xaxis_title="x",
    yaxis_title="y",
    legend=dict(x=0, y=1),
    plot_bgcolor='white',  # White background
    paper_bgcolor='white'  # White background
)

fig.show()

Epoch: 0, Loss: 3.07454252243042
Epoch: 10, Loss: 2.583817958831787
Epoch: 20, Loss: 2.3773999214172363
Epoch: 30, Loss: 2.298490285873413
Epoch: 40, Loss: 2.2663278579711914
Epoch: 49, Loss: 2.234740972518921


In [21]:
import torch
from torch import nn
import plotly.graph_objects as go
from edl_pytorch import NormalInvGamma
from edl_pytorch.loss import nig_nll, nig_reg


torch.manual_seed(0)

# Define the true function
true_func = lambda x: torch.sin(2 * x) * x 

x_train, y_train, x_test, y_test, outlier_indices = generate_data(
    lambda x: torch.sin(2 * x) * x ,
    0,
    2 * torch.pi,
    num_train=50,
    test_additional=0.2,
    num_test=1000,
    sigma_scale=3,
    outlier_ratio=0.04,
    outlier_value_range=(10, 10.1),
)

# Create Evidential MLP model
model = EvidentialMLP(
    input_dim=1, 
    output_dim=1,
    hidden_units=[256, 256, 256],
    activations=["tanh", "tanh", "tanh"],
    nig_coeff=1e-2,
    lipschitz_coeff=0,
    nsu_coeff=0,
    epoch_num=5000,
    lr=5e-4
)

# Train the model
model.fit(x_train, y_train)

# Make predictions on test data
mean, std = model(x_test)

# Prepare data for visualization
mean, std = mean.squeeze().numpy(), std.squeeze().numpy()
x_test, y_test = x_test.squeeze().numpy(), y_test.squeeze().numpy()

# Plot using Plotly
fig = go.Figure()

# Add train data points
fig.add_trace(go.Scatter(x=x_train.squeeze().numpy(), y=y_train.squeeze().numpy(), mode='markers', 
                         marker=dict(size=3, color='blue'), name='Train'))

# Add true function line
fig.add_trace(go.Scatter(x=x_test, y=y_test, mode='lines', 
                         line=dict(color='black', width=2), name='True'))

# Add predicted mean line
fig.add_trace(go.Scatter(x=x_test, y=mean, mode='lines', 
                         line=dict(color='blue', dash='dash'), name='Pred'))

# Add uncertainty (2 std devs only)
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean - 2 * std), 
    fill=None, mode='lines', line_color='rgba(0,0,255,0.1)',  # Lighter color
    showlegend=True, name='Unc. (2σ)'
))
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean + 2 * std), 
    fill='tonexty', mode='lines', line_color='rgba(0,0,255,0.1)',
    showlegend=False
))

# Update layout with white background
fig.update_layout(
    title="Evidential Regression",
    xaxis_title="x",
    yaxis_title="y",
    legend=dict(x=0, y=1),
    plot_bgcolor='white',  # White background
    paper_bgcolor='white'  # White background
)

fig.show()

Epoch: 0, Loss: 3.9521982669830322
Epoch: 10, Loss: 3.027963638305664
Epoch: 20, Loss: 2.9821746349334717
Epoch: 30, Loss: 2.954984426498413
Epoch: 40, Loss: 2.935110569000244
Epoch: 50, Loss: 2.9156723022460938
Epoch: 60, Loss: 2.8983147144317627
Epoch: 70, Loss: 2.8805363178253174
Epoch: 80, Loss: 2.862321615219116
Epoch: 90, Loss: 2.8442060947418213
Epoch: 100, Loss: 2.8305063247680664
Epoch: 110, Loss: 2.8203070163726807
Epoch: 120, Loss: 2.8109116554260254
Epoch: 130, Loss: 2.8014657497406006
Epoch: 140, Loss: 2.791114330291748
Epoch: 150, Loss: 2.7780189514160156
Epoch: 160, Loss: 2.7590019702911377
Epoch: 170, Loss: 2.7393500804901123
Epoch: 180, Loss: 2.725006103515625
Epoch: 190, Loss: 2.69647216796875
Epoch: 200, Loss: 2.695117950439453
Epoch: 210, Loss: 2.669711112976074
Epoch: 220, Loss: 2.6688833236694336
Epoch: 230, Loss: 2.6914725303649902
Epoch: 240, Loss: 2.6716225147247314
Epoch: 250, Loss: 2.640979766845703
Epoch: 260, Loss: 2.6967263221740723
Epoch: 270, Loss: 2.660

In [51]:
import torch
from torch import nn
import plotly.graph_objects as go
from edl_pytorch import NormalInvGamma
from edl_pytorch.loss import nig_nll, nig_reg


torch.manual_seed(0)

# Define the true function
true_func = lambda x: 2*torch.sin(5 * x) * x 

x_train, y_train, x_test, y_test, outlier_indices = generate_data(
    lambda x: torch.sin(2 * x) * x ,
    0,
    2 * torch.pi,
    num_train=50,
    test_additional=0.2,
    num_test=1000,
    sigma_scale=1,
    outlier_ratio=0.04,
    outlier_value_range=(10, 10.1),
)

# Create Evidential MLP model
model = EvidentialMLP(
    input_dim=1, 
    output_dim=1,
    hidden_units=[256, 256, 256],
    activations=["tanh", "tanh", "tanh"],
    nig_coeff=1e-2,
    lipschitz_coeff=0,
    nsu_coeff=0,
    epoch_num=500,
    lr=5e-4
)

# Train the model
model.fit(x_train, y_train)

# Make predictions on test data
mean, std = model(x_test)

# Prepare data for visualization
mean, std = mean.squeeze().numpy(), std.squeeze().numpy()
x_test, y_test = x_test.squeeze().numpy(), y_test.squeeze().numpy()

# Plot using Plotly
fig = go.Figure()

# Add train data points
fig.add_trace(go.Scatter(x=x_train.squeeze().numpy(), y=y_train.squeeze().numpy(), mode='markers', 
                         marker=dict(size=3, color='blue'), name='Train'))

# Add true function line
fig.add_trace(go.Scatter(x=x_test, y=y_test, mode='lines', 
                         line=dict(color='black', width=2), name='True'))

# Add predicted mean line
fig.add_trace(go.Scatter(x=x_test, y=mean, mode='lines', 
                         line=dict(color='blue', dash='dash'), name='Pred'))

# Add uncertainty (2 std devs only)
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean - 2 * std), 
    fill=None, mode='lines', line_color='rgba(0,0,255,0.1)',  # Lighter color
    showlegend=True, name='Unc. (2σ)'
))
fig.add_trace(go.Scatter(
    x=x_test, 
    y=(mean + 2 * std), 
    fill='tonexty', mode='lines', line_color='rgba(0,0,255,0.1)',
    showlegend=False
))

# Update layout with white background
fig.update_layout(
    title="Evidential Regression",
    xaxis_title="x",
    yaxis_title="y",
    legend=dict(x=0, y=1),
    plot_bgcolor='white',  # White background
    paper_bgcolor='white'  # White background
)

fig.show()

Epoch: 0, Loss: 2.986140727996826
Epoch: 10, Loss: 2.4654486179351807
Epoch: 20, Loss: 2.4217193126678467
Epoch: 30, Loss: 2.3927175998687744
Epoch: 40, Loss: 2.369886875152588
Epoch: 50, Loss: 2.341498851776123
Epoch: 60, Loss: 2.305126190185547
Epoch: 70, Loss: 2.282029628753662
Epoch: 80, Loss: 2.2684407234191895
Epoch: 90, Loss: 2.2562854290008545
Epoch: 100, Loss: 2.2455291748046875
Epoch: 110, Loss: 2.2511982917785645
Epoch: 120, Loss: 2.237234115600586
Epoch: 130, Loss: 2.229703903198242
Epoch: 140, Loss: 2.225137710571289
Epoch: 150, Loss: 2.2200944423675537
Epoch: 160, Loss: 2.2140676975250244
Epoch: 170, Loss: 2.206270694732666
Epoch: 180, Loss: 2.1982879638671875
Epoch: 190, Loss: 2.2084670066833496
Epoch: 200, Loss: 2.1867048740386963
Epoch: 210, Loss: 2.1715784072875977
Epoch: 220, Loss: 2.1541476249694824
Epoch: 230, Loss: 2.1480777263641357
Epoch: 240, Loss: 2.1159415245056152
Epoch: 250, Loss: 2.119462013244629
Epoch: 260, Loss: 2.093606472015381
Epoch: 270, Loss: 2.057

In [ ]:
# MLP 以外のモデルを使ってみる